# Array arithmetic with scalars and other arrays

This notebook demonstrates arithmetic operations on `pyclesperanto.Array`, using standard
Python operators, either with a scalar constant or with another array (element-wise, with
broadcasting support).

In [1]:
import numpy as np
import pyclesperanto as cle

cle.select_device()

(OpenCL) Apple M5 Max (OpenCL 1.2)
	Vendor:                      Apple
	Driver Version:              1.2 1.0
	Device Type:                 GPU
	Compute Units:               40
	Global Memory Size:          53084 MB
	Local Memory Size:           0 MB
	Maximum Buffer Size:         9953 MB
	Max Clock Frequency:         1000 MHz
	Image Support:               Yes

## 1. Array and scalar

Arithmetic operators work directly between an array and a scalar.

In [2]:
arr = cle.push(np.arange(1, 11, dtype=np.float32))
arr

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.], dtype=float32, mtype=buffer)

In [3]:
arr + 10, arr - 5, arr * 2, arr / 4

(array([11., 12., 13., 14., 15., 16., 17., 18., 19., 20.], dtype=float32, mtype=buffer),
 array([-4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.], dtype=float32, mtype=buffer),
 array([ 2.,  4.,  6.,  8., 10., 12., 14., 16., 18., 20.], dtype=float32, mtype=buffer),
 array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 ],
       dtype=float32, mtype=buffer))

In [4]:
arr // 3, arr % 3, arr ** 2

(array([0., 0., 1., 1., 1., 2., 2., 2., 3., 3.], dtype=float32, mtype=buffer),
 array([1., 2., 0., 1., 2., 0., 1., 2., 0., 1.], dtype=float32, mtype=buffer),
 array([  1.,   4.,   9.,  16.,  25.,  36.,  49.,  64.,  81., 100.],
       dtype=float32, mtype=buffer))

Scalars can also be on the left-hand side (reflected operators).

In [5]:
10 - arr, 100 / arr, 2 ** arr

(array([9., 8., 7., 6., 5., 4., 3., 2., 1., 0.], dtype=float32, mtype=buffer),
 array([100.      ,  50.      ,  33.333332,  25.      ,  20.      ,
         16.666666,  14.285714,  12.5     ,  11.111111,  10.      ],
       dtype=float32, mtype=buffer),
 array([   2.,    4.,    8.,   16.,   32.,   64.,  128.,  256.,  512.,
        1024.], dtype=float32, mtype=buffer))

In-place operators (`+=`, `-=`, `*=`, `/=`, ...) modify the array on the device
without creating a new one.

In [6]:
arr_copy = arr.copy()
arr_copy += 1
arr_copy *= 2
arr_copy

array([ 4.,  6.,  8., 10., 12., 14., 16., 18., 20., 22.], dtype=float32, mtype=buffer)

## 2. Array and array

The same operators work between two arrays of the same shape, element-wise.

In [7]:
a = cle.push(np.asarray([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32))
b = cle.push(np.asarray([2.0, 3.0, 1.0, 5.0, 2.0], dtype=np.float32))

a + b, a - b, a * b, a / b

(array([3., 5., 4., 9., 7.], dtype=float32, mtype=buffer),
 array([-1., -1.,  2., -1.,  3.], dtype=float32, mtype=buffer),
 array([ 2.,  6.,  3., 20., 10.], dtype=float32, mtype=buffer),
 array([0.5      , 0.6666667, 3.       , 0.8      , 2.5      ],
       dtype=float32, mtype=buffer))

In [8]:
a // b, a % b, a ** b

(array([0., 0., 3., 0., 2.], dtype=float32, mtype=buffer),
 array([1., 2., 0., 4., 1.], dtype=float32, mtype=buffer),
 array([1.000e+00, 8.000e+00, 3.000e+00, 1.024e+03, 2.500e+01],
       dtype=float32, mtype=buffer))

## 3. Broadcasting

As in NumPy, arrays with compatible shapes (matching or size-1 dimensions) are broadcast
against each other.

In [9]:
matrix = cle.push(np.arange(12, dtype=np.float32).reshape(3, 4))
row = cle.push(np.asarray([[10.0, 20.0, 30.0, 40.0]], dtype=np.float32))

matrix + row

array([[10., 21., 32., 43.],
       [14., 25., 36., 47.],
       [18., 29., 40., 51.]], dtype=float32, mtype=buffer)

In [10]:
column = cle.push(np.asarray([[1.0], [2.0], [3.0]], dtype=np.float32))
matrix * column

array([[ 0.,  1.,  2.,  3.],
       [ 8., 10., 12., 14.],
       [24., 27., 30., 33.]], dtype=float32, mtype=buffer)

## 4. Mixing with NumPy arrays

A `pyclesperanto.Array` can be combined directly with a plain NumPy array; the NumPy array is
pushed to the device automatically.

In [11]:
np_array = np.asarray([1.0, 1.0, 1.0, 1.0, 1.0], dtype=np.float32)
a + np_array

array([2., 3., 4., 5., 6.], dtype=float32, mtype=buffer)

## 5. Arithmetics optmization wiht `evaluate()` function

Each operation call a kernel on the GPU which can create a slowdown on the computation when performing equations like ```f = (cos(A)² + sin(B)²) / C```.

In such case, we advise to rely on the ```evaluate()``` function that allows you to pass an equation directly into one kernel call.

> Some operators are not compatible, such as `^` or `**`, instead use their function equivalent `pow()`, `cos()`, `sin()`, `log()`.
> Only element-wise operation can be performed, reduction operation must be computed before the ```evaluate()```

In [12]:
cle.evaluate("(pow(cos(a), 2) + pow(sin(b), 2)) * c", {"a": a, "b": b, "c": 3.0})

array([3.356245 , 0.5792792, 5.064476 , 4.0403576, 2.721858 ],
      dtype=float32, mtype=buffer)

a factor 5 can be gain that way

In [13]:
import time

cle.wait_for_kernel_to_finish()  # needed for accurate timing

# Create test arrays
array_A = cle.push(np.ones((1000, 1000)))
array_B = cle.push(np.ones((1000, 1000)))
constant = 10

# Method 1: Using multiple operations (multiple kernels)
start = time.time()
result_multi = (cle.power(cle.cos(array_A), scalar=2.0) + cle.power(cle.sin(array_B), scalar=2.0)) * constant
end = time.time()
multi_kernel_time = end - start
print(f"Multiple kernels: {multi_kernel_time:.6f} seconds")

# Method 2: Using evaluate (single kernel)
start = time.time()
result_single = cle.evaluate("(pow(cos(a), 2) + pow(sin(b), 2)) * c", 
                              {"a": array_A, "b": array_B, "c": constant})
end = time.time()
one_kernel_time = end - start
print(f"Single kernel (evaluate): {one_kernel_time:.6f} seconds")

# Compare performance
speedup = multi_kernel_time / one_kernel_time
print(f"\nevaluate() is {speedup:.1f}x faster")

Multiple kernels: 0.154741 seconds
Single kernel (evaluate): 0.028829 seconds

evaluate() is 5.4x faster
